# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MihirJayswal812007/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. Unit of Analysis: One row represents a single anonymized webpage's performance over a specific month.
2. Time Window: I am using a mid-panel month, specifically month = '2026-03', to ensure I treat the final month of the dataset as a sealed test set and avoid outcome leakage.
3. Table Used: The main internship-warehouse dataset from Hugging Face.

In [7]:
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Load HF Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect to DuckDB & pass the token directly to it securely
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. CORRECTED Path to the warehouse
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Query 1: Verify the grain and row count
query_grain = f"SELECT count(*) as total_rows FROM '{hf_path}'"
display(con.execute(query_grain).df())

,total_rows
0,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label (Proxy): needs_redesign. This is a binary label (1 if the page ranks on Page 1 but has a CTR < 0.05, else 0).
Excluded: I am excluding pages with 0 impressions. If a page is entirely invisible in search, it is an SEO issue, not a UI/UX layout issue.
My 5 Features:
1.content_type: Knowable at the decision moment because the format is static in the CMS.
2.word_count: Knowable at the decision moment because the text length is established.
3.content_age_days: Knowable at the decision moment because the publication date is fixed.
4.days_since_last_update: Knowable at the decision moment from revision logs.
5.position_tier: Knowable at the decision moment based on historical average ranking brackets.

In [11]:
# Query 2: Build the slice with our exact features, label, and exclusion rule
query_features = f"""
SELECT
    d.content_type,
    d.word_count,
    d.position_tier,
    f.ctr,
    f.gsc_impressions,
    CASE WHEN d.position_tier = 'page_1' AND f.ctr < 0.05 THEN 1 ELSE 0 END as needs_redesign
FROM '{hf_path}' AS f
JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' AS d
  ON f.content_hash_id = d.content_hash_id
WHERE f.gsc_impressions > 0
"""

df = con.execute(query_features).df()
print("Data slice loaded successfully. First 5 rows:")
display(df.head())

BinderException: Binder Error: Table "f" does not have a column named "position_tier"

Candidate bindings: : "ai_other"

LINE 5:     f.position_tier,
            ^

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

BinderException: Binder Error: Referenced column "impressions_90d" not found in FROM clause!
Candidate bindings: "sessions_paid", "sessions_ai", "gsc_impressions", "sessions_direct", "sessions_organic"

LINE 12: WHERE impressions_90d > 0
               ^

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.